# Medical Debate System - Integration Test
This notebook tests the full pipeline: Retrieval → Debate (Doctor A vs Doctor B).
Requires `OPENAI_API_KEY` set in a `.env` file at the project root.

In [ ]:
import sys, os

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)
os.chdir(project_root)

from dotenv import load_dotenv
load_dotenv()

print("Project root:", project_root)
print("API key set:", bool(os.getenv("OPENAI_API_KEY")))

## 1. Build Retrieval Pipeline (Person 1's code)

In [ ]:
from src.retrieval.load_dataset import load_pubmedqa
from src.retrieval.chunk_documents import chunk_documents
from src.retrieval.create_embeddings import load_embedding_model, create_embeddings
from src.retrieval.vector_store import create_chroma_collection, store_documents
from src.retrieval.bm25_index import build_bm25_index
from src.retrieval.reranker import load_reranker
from src.retrieval.retrieve_evidence import retrieve_evidence_hybrid_reranked

dataset = load_pubmedqa(limit=200)
documents = chunk_documents(dataset)
print(f"Loaded {len(documents)} documents")

model = load_embedding_model()
embeddings = create_embeddings(model, documents)
print(f"Embeddings shape: {embeddings.shape}")

bm25 = build_bm25_index(documents)
reranker = load_reranker()

collection = create_chroma_collection(
    collection_name="pubmedqa_debate_test",
    persist_directory="./chroma_store_debate_test"
)
if collection.count() == 0:
    store_documents(collection, documents, embeddings=embeddings)

print(f"ChromaDB collection ready: {collection.count()} docs")

## 2. Test Retrieval on a Sample Question

In [ ]:
sample_question = documents[0]["question"]
print("Question:", sample_question)
print("Ground truth:", documents[0]["final_decision"])
print()

evidence = retrieve_evidence_hybrid_reranked(
    collection=collection,
    model=model,
    bm25=bm25,
    documents=documents,
    reranker=reranker,
    query=sample_question,
    top_k=3,
    candidate_k=10
)

for i, item in enumerate(evidence, 1):
    print(f"Evidence {i}: (rerank={item.get('rerank_score', 0):.2f})")
    print(f"  Question: {item['metadata']['question'][:100]}")
    print()

## 3. Run a Single Debate

In [ ]:
from src.debate import run_debate

transcript = run_debate(
    question=sample_question,
    evidence=evidence,
    max_rounds=2
)

print(f"Debate completed: {transcript.num_rounds} rounds, {len(transcript.messages)} messages")
print(f"Doctor A final: {transcript.doctor_a_final_position} (confidence: {transcript.doctor_a_final_confidence})")
print(f"Doctor B final: {transcript.doctor_b_final_position} (confidence: {transcript.doctor_b_final_confidence})")

In [ ]:
# Print full debate transcript
for msg in transcript.messages:
    print("=" * 60)
    print(f"[Round {msg.round}] {msg.agent.upper()}")
    print(f"Position: {msg.position} | Confidence: {msg.confidence}")
    print(f"Evidence cited: {msg.evidence_cited}")
    print(f"Reasoning: {msg.reasoning[:500]}...")
    print()

## 4. Run Debates on Multiple Questions

In [ ]:
import json

test_questions = [
    {"question": documents[i]["question"], "ground_truth": documents[i]["final_decision"]}
    for i in range(5)
]

results = []
for i, item in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"Question {i+1}: {item['question'][:80]}...")
    print(f"Ground truth: {item['ground_truth']}")

    ev = retrieve_evidence_hybrid_reranked(
        collection=collection, model=model, bm25=bm25,
        documents=documents, reranker=reranker,
        query=item["question"], top_k=3, candidate_k=10
    )

    t = run_debate(question=item["question"], evidence=ev, max_rounds=2)
    t.metadata["ground_truth"] = item["ground_truth"]

    print(f"Doctor A: {t.doctor_a_final_position} ({t.doctor_a_final_confidence:.2f})")
    print(f"Doctor B: {t.doctor_b_final_position} ({t.doctor_b_final_confidence:.2f})")

    results.append(t.model_dump())

print(f"\nCompleted {len(results)} debates")

## 5. Save Transcripts for Person 3 (Judge)

In [ ]:
os.makedirs("outputs", exist_ok=True)

with open("outputs/debate_transcripts.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} transcripts to outputs/debate_transcripts.json")